In [ ]:
def generate_chain(device_id, start_timestamp, n_steps=1, mode='with_gps',
                   lat=None, lon=None, positions=None):
    start_ts = pd.Timestamp(start_timestamp)
    if start_ts.tzinfo is None:
        start_ts = start_ts.tz_localize('Europe/Berlin')

    device_col = f'device_{device_id}'
    device_data = df[df[device_col] == 1].sort_values('timestamp').reset_index(drop=True)

    mask = device_data['timestamp'] < start_ts
    if mask.sum() < SEQ_LEN:
        print(f"Error: Need {SEQ_LEN} rows before {start_ts}, found {mask.sum()}")
        return None

    seed_idx = device_data[mask].index[-SEQ_LEN:]
    seed = device_data.loc[seed_idx].reset_index(drop=True)

    # DON'T build positions list — let model predict GPS autoregressively
    # Only inject the FIRST lat/lon into the seed if provided

    base_buffer = {}
    for col in BASE_INPUTS:
        base_buffer[col] = list(seed[col].values)

    gen_buffer = {}
    for col in ALL_GENERATED_COLS:
        gen_buffer[col] = list(seed[col].values)

    # If starting lat/lon provided, override the last seed position
    # so the model starts predicting from this location
    if lat is not None and lon is not None:
        gen_buffer['Latitude'][-1] = lat
        gen_buffer['Longitude'][-1] = lon

    generated_rows = []

    for step in range(n_steps):
        current_ts = start_ts + pd.Timedelta(seconds=step)

        current_base = {
            'hour': current_ts.hour,
            'day_of_week': current_ts.dayofweek + 1,
            'device_pc1': 1 if device_id == 'pc1' else 0,
            'device_pc2': 1 if device_id == 'pc2' else 0,
            'device_pc3': 1 if device_id == 'pc3' else 0,
            'device_pc4': 1 if device_id == 'pc4' else 0,
            'direction_downlink': base_buffer['direction_downlink'][-1],
            'direction_uplink': base_buffer['direction_uplink'][-1],
            'measured_qos_datarate': base_buffer['measured_qos_datarate'][-1],
            'measured_qos_delay': base_buffer['measured_qos_delay'][-1],
            'measurement': base_buffer['measurement'][-1],
            'operator': base_buffer['operator'][-1],
        }

        for col in BASE_INPUTS:
            base_buffer[col].append(current_base[col])
        for col in ALL_GENERATED_COLS:
            gen_buffer[col].append(0.0)

        for level in CAUSAL_CHAIN:
            level_name = level['name']
            context_cols = BASE_INPUTS + level['extra_inputs']
            auto_cols = level['auto_inputs']
            target_cols = level['targets']

            model_level = chain_models[level_name]
            scaler_in = chain_scalers[f"{level_name}_input"]
            scaler_tgt = chain_scalers[f"{level_name}_target"]

            # NO SKIPPING GPS — let model predict it every step
            # The seed has the starting lat/lon, model predicts forward

            ctx_seq = np.zeros((SEQ_LEN, len(context_cols)), dtype=np.float32)
            auto_seq = np.zeros((SEQ_LEN, len(auto_cols)), dtype=np.float32)

            for t in range(SEQ_LEN):
                buf_idx = len(base_buffer['hour']) - SEQ_LEN + t
                for j, col in enumerate(context_cols):
                    if col in BASE_INPUTS:
                        ctx_seq[t, j] = base_buffer[col][buf_idx]
                    else:
                        ctx_seq[t, j] = gen_buffer[col][buf_idx]
                for j, col in enumerate(auto_cols):
                    auto_seq[t, j] = gen_buffer[col][buf_idx]

            auto_seq[-1, :] = 0.0
            x = np.concatenate([ctx_seq, auto_seq], axis=1)
            x_scaled = scaler_in.transform(x.reshape(-1, x.shape[1])).reshape(1, SEQ_LEN, -1)

            with torch.no_grad():
                pred_scaled = model_level(
                    torch.FloatTensor(x_scaled).to(device_torch)
                ).cpu().numpy()
            pred = scaler_tgt.inverse_transform(pred_scaled)[0]

            for j, col in enumerate(target_cols):
                val = pred[j]
                if col in SNAP_RULES:
                    val = snap_to_nearest(val, SNAP_RULES[col])
                gen_buffer[col][-1] = val

        # NO OVERRIDE of lat/lon — keep what the model predicted

        row = {'timestamp': current_ts}
        for col in BASE_INPUTS:
            row[col] = current_base[col]
        for col in ALL_GENERATED_COLS:
            row[col] = gen_buffer[col][-1]
        row['COG'] = np.degrees(np.arctan2(row['sin_COG'], row['cos_COG'])) % 360
        row['jitter'] = np.expm1(row['jitter_log'])
        generated_rows.append(row)

    return pd.DataFrame(generated_rows)


In [ ]:
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

def plot_day_comparison(device_id, day_index=0, gen_every=100):
    """
    Plot real vs generated for an entire day (first half).
    Generates at every gen_every-th row and overlays on real data.
    """
    device_data = df[df[f'device_{device_id}'] == 1].sort_values('timestamp').reset_index(drop=True)
    dates = sorted(device_data['timestamp'].dt.date.unique())

    if day_index >= len(dates):
        print(f"Only {len(dates)} days available")
        return

    day = device_data[device_data['timestamp'].dt.date == dates[day_index]].reset_index(drop=True)
    print(f"Day {day_index+1} ({dates[day_index]}): {len(day)} rows")

    # First half only
    half_idx = len(day) // 2
    day = day.iloc[:half_idx].reset_index(drop=True)
    print(f"Using first half: {len(day)} rows")

    # Find segments
    gaps = day['timestamp'].diff().dt.total_seconds()
    break_points = gaps[gaps > 60].index.tolist()
    starts = [0] + break_points
    ends = break_points + [len(day)]

    segments = []
    for s, e in zip(starts, ends):
        seg = day.iloc[s:e].reset_index(drop=True)
        if len(seg) > SEQ_LEN + 10:
            segments.append(seg)

    print(f"Segments: {len(segments)}")
    print(f"Generating at every {gen_every}th row...")

    # Generate at sampled points
    gen_points = []
    for seg_idx, seg in enumerate(segments):
        for idx in range(SEQ_LEN + 5, len(seg) - 5, gen_every):
            row = seg.iloc[idx]
            ts = str(row['timestamp'])

            gen = generate_chain(device_id, ts, n_steps=1, mode='with_gps',
                                 lat=row['Latitude'], lon=row['Longitude'])

            if gen is not None and len(gen) > 0:
                gen_points.append({
                    'timestamp': row['timestamp'],
                    'seg_idx': seg_idx,
                    'real': {col: float(row[col]) for col in row.index
                             if col != 'timestamp' and isinstance(row[col], (int, float, np.integer, np.floating))},
                    'gen': gen.iloc[0].to_dict(),
                })

    print(f"Generated {len(gen_points)} comparison points")

    if len(gen_points) == 0:
        return

    features = [
        ('PCell_RSRP_max', 'RSRP (dBm)'),
        ('datarate', 'Datarate (log)'),
        ('ping_ms', 'Ping (log ms)'),
        ('speed_kmh', 'Speed (km/h)'),
        ('PCell_SNR_1', 'SNR (dB)'),
        ('PCell_freq_MHz', 'Frequency (MHz)'),
    ]

    fig, axes = plt.subplots(len(features), 1,
                             figsize=(20, 4 * len(features)),
                             facecolor=BG)
    fig.suptitle(
        f'Hybrid GAN — Day {day_index+1} ({dates[day_index]}) | Device {device_id}\n'
        f'Real (blue) vs Generated (red) — sampled every {gen_every} rows',
        color=TEXT, fontsize=16, fontweight='bold', y=1.01
    )

    for feat_idx, (col, label) in enumerate(features):
        ax = axes[feat_idx]
        ax.set_facecolor(CARD)
        ax.tick_params(colors=MUTED, labelsize=9)
        for sp in ax.spines.values():
            sp.set_edgecolor(BORDER)
        ax.grid(True, alpha=0.1, color=MUTED)

        # Plot per segment
        for seg_idx in sorted(set(gp['seg_idx'] for gp in gen_points)):
            seg_gps = sorted(
                [gp for gp in gen_points if gp['seg_idx'] == seg_idx],
                key=lambda x: x['timestamp']
            )

            times = [pd.Timestamp(gp['timestamp']).tz_localize(None) for gp in seg_gps]
            real_vals = [float(gp['real'].get(col, np.nan)) for gp in seg_gps]
            gen_vals = [float(gp['gen'].get(col, np.nan)) for gp in seg_gps]

            # Real — solid blue
            ax.plot(times, real_vals, color=BLUE, linewidth=2.0,
                    alpha=0.85, label='Real' if seg_idx == 0 else '')

            # Generated — red
            ax.plot(times, gen_vals, color=RED, linewidth=1.8,
                    alpha=0.85, label='Generated' if seg_idx == 0 else '')

            # Error shading between real and generated
            ax.fill_between(times, real_vals, gen_vals,
                           alpha=0.08, color=ORANGE)

            # RMSE for first segment
            if seg_idx == 0:
                valid = [(r, g) for r, g in zip(real_vals, gen_vals)
                         if not np.isnan(r) and not np.isnan(g)]
                if valid:
                    rs, gs = zip(*valid)
                    rmse = np.sqrt(np.mean((np.array(rs) - np.array(gs))**2))
                    ax.text(0.98, 0.95, f'RMSE: {rmse:.2f}',
                            transform=ax.transAxes, fontsize=10,
                            ha='right', va='top', color=TEXT,
                            bbox=dict(boxstyle='round,pad=0.3',
                                     facecolor=CARD, edgecolor=BORDER, alpha=0.9))

        # Segment boundaries
        for seg_idx, seg in enumerate(segments):
            if seg_idx > 0:
                boundary = pd.Timestamp(seg['timestamp'].iloc[0]).tz_localize(None)
                ax.axvline(x=boundary, color=MUTED, linestyle='--',
                          linewidth=1, alpha=0.4)

        ax.set_ylabel(label, color=TEXT, fontsize=11, fontweight='bold')
        if feat_idx == 0:
            real_patch = mpatches.Patch(color=BLUE, alpha=0.85, label='Real')
            gen_patch = mpatches.Patch(color=RED, alpha=0.85, label='Generated')
            ax.legend(handles=[real_patch, gen_patch],
                     framealpha=0.2, labelcolor=TEXT, fontsize=10, loc='upper left')

    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    axes[-1].set_xlabel('Time', color=MUTED, fontsize=11)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(f'day_comparison_{device_id}_day{day_index+1}.png',
                dpi=150, facecolor=BG, bbox_inches='tight')
    plt.show()

    # Print summary
    print(f"\nSummary — {device_id} Day {day_index+1}:")
    for col, label in features:
        valid = [(float(gp['real'].get(col, np.nan)),
                  float(gp['gen'].get(col, np.nan))) for gp in gen_points]
        valid = [(r, g) for r, g in valid if not np.isnan(r) and not np.isnan(g)]
        if valid:
            rs, gs = zip(*valid)
            rmse = np.sqrt(np.mean((np.array(rs) - np.array(gs))**2))
            mae = np.mean(np.abs(np.array(rs) - np.array(gs)))
            corr = np.corrcoef(rs, gs)[0, 1] if len(rs) > 2 else 0
            print(f"  {label:<20} RMSE: {rmse:>8.3f}  MAE: {mae:>8.3f}  Corr: {corr:>6.3f}")


# ============================================================
# RUN — All 4 devices, Day 1
# ============================================================
BG = '#0d1117'
CARD = '#161b22'
BORDER = '#30363d'
TEXT = '#e6edf3'
MUTED = '#8b949e'
GREEN = '#3fb950'
BLUE = '#58a6ff'
RED = '#ff7b72'
ORANGE = '#f0883e'



In [ ]:
plot_day_comparison('pc1', day_index=0, gen_every=50)

In [ ]:
import matplotlib.dates as mdates
import matplotlib.patches as mpatches

BG = '#0d1117'
CARD = '#161b22'
BORDER = '#30363d'
TEXT = '#e6edf3'
MUTED = '#8b949e'
GREEN = '#3fb950'
BLUE = '#58a6ff'
RED = '#ff7b72'
ORANGE = '#f0883e'

def style_ax(ax):
    ax.set_facecolor(CARD)
    ax.tick_params(colors=MUTED, labelsize=8)
    for sp in ax.spines.values():
        sp.set_edgecolor(BORDER)
    ax.grid(True, alpha=0.1, color=MUTED)

def find_gaps(device_id, day_index=0, min_gap_sec=60, max_gap_sec=14400):
    """Find all gaps for a device on a specific day."""
    device_col = f'device_{device_id}'
    device_data = df[df[device_col] == 1].sort_values('timestamp').reset_index(drop=True)
    dates = sorted(device_data['timestamp'].dt.date.unique())

    if day_index >= len(dates):
        print(f"Only {len(dates)} days available")
        return None, None, None

    day_date = dates[day_index]
    day = device_data[device_data['timestamp'].dt.date == day_date].reset_index(drop=True)

    print(f"Device {device_id} — Day {day_index+1} ({day_date})")
    print(f"Total rows: {len(day)}")
    print(f"Time range: {day['timestamp'].iloc[0]} → {day['timestamp'].iloc[-1]}")

    # Find gaps
    deltas = day['timestamp'].diff().dt.total_seconds()
    gaps = []

    for idx in deltas[deltas > min_gap_sec].index:
        gap_sec = deltas[idx]
        if gap_sec > max_gap_sec:
            continue

        before_row = day.iloc[idx - 1]
        after_row = day.iloc[idx]

        gaps.append({
            'gap_id': len(gaps),
            'gap_start': before_row['timestamp'],
            'gap_end': after_row['timestamp'],
            'duration_sec': int(gap_sec),
            'duration_min': gap_sec / 60,
            'before_idx': idx - 1,
            'after_idx': idx,
            'before_lat': before_row['Latitude'],
            'before_lon': before_row['Longitude'],
            'before_speed': before_row['speed_kmh'],
            'after_lat': after_row['Latitude'],
            'after_lon': after_row['Longitude'],
            'after_speed': after_row['speed_kmh'],
        })

    # Print gap table
    print(f"\nFound {len(gaps)} gaps (>{min_gap_sec}s, <{max_gap_sec}s):\n")
    print(f"  {'ID':>4} {'Start':>22} {'End':>22} {'Duration':>10} {'Before Speed':>14} {'After Speed':>13}")
    print(f"  {'─'*90}")
    for g in gaps:
        print(f"  {g['gap_id']:>4} "
              f"{str(g['gap_start'])[:22]:>22} "
              f"{str(g['gap_end'])[:22]:>22} "
              f"{g['duration_sec']:>7}s ({g['duration_min']:.1f}m) "
              f"{g['before_speed']:>10.1f} km/h "
              f"{g['after_speed']:>10.1f} km/h")

    

    # Find segments
    seg_gaps = day['timestamp'].diff().dt.total_seconds()
    break_points = seg_gaps[seg_gaps > min_gap_sec].index.tolist()
    starts = [0] + break_points
    ends = break_points + [len(day)]


    return day, gaps, day_date







In [ ]:
day_data, gaps, day_date = find_gaps('pc1', day_index=0)

In [ ]:

def fill_gap(device_id, day_data, gap, context_before=400, context_after=50):
    """Fill a specific gap using generate_chain."""
    print(f"\nFilling Gap {gap['gap_id']}:")
    print(f"  Start: {gap['gap_start']}")
    print(f"  End:   {gap['gap_end']}")
    print(f"  Duration: {gap['duration_sec']}s ({gap['duration_min']:.1f} min)")

    start_ts = gap['gap_start'] + pd.Timedelta(seconds=1)
    n_steps = gap['duration_sec'] - 1

    if n_steps > 3600:
        print(f"  Gap too long ({n_steps}s). Limiting to 3600s.")
        n_steps = 3600

    print(f"  Generating {n_steps} timesteps...")

    gen = generate_chain(
        device_id, str(start_ts), n_steps=n_steps,
        mode='with_gps',
        lat=gap['before_lat'], lon=gap['before_lon']
    )

    if gen is None:
        print("  Generation failed!")
        return None

    print(f"  Generated {len(gen)} rows")

    # Get context: real data before and after gap
    before_start = max(0, gap['before_idx'] - context_before)
    before = day_data.iloc[before_start:gap['before_idx'] + 1]

    after_end = min(len(day_data), gap['after_idx'] + context_after)
    after = day_data.iloc[gap['after_idx']:after_end]

    result = {
        'gap': gap,
        'generated': gen,
        'before': before,
        'after': after,
    }

    return result

In [ ]:


def plot_filled_gap(fill_result, features=None, show_before=400, show_after=50):
    """Plot real data before/after with synthetic fill in between."""
    if fill_result is None:
        print("No fill result to plot")
        return

    gap = fill_result['gap']
    gen = fill_result['generated']
    before = fill_result['before'].tail(show_before)
    after = fill_result['after'].head(show_after)

    if features is None:
        features = [
            ('PCell_RSRQ_max', 'RSRQ (dBm)'),
            ('datarate', 'Datarate (log)'),
            ('ping_ms', 'Ping (log ms)'),
            ('PCell_SNR_1', 'SNR (dB)'),
            ('PCell_freq_MHz', 'Frequency (MHz)'),
        ]

    n_before = len(before)
    n_gen = len(gen)
    n_after = len(after)

    fig, axes = plt.subplots(len(features), 1,
                             figsize=(18, 3.5 * len(features)),
                             facecolor=BG)
    fig.suptitle(
        f'Gap Fill — Gap {gap["gap_id"]} | '
        f'{str(gap["gap_start"])[:19]} → {str(gap["gap_end"])[:19]} | '
        f'{gap["duration_sec"]}s ({gap["duration_min"]:.1f} min)\n'
        f'Before: {n_before} rows | Generated: {n_gen} rows | After: {n_after} rows',
        color=TEXT, fontsize=14, fontweight='bold', y=1.02
    )

    for i, (col, label) in enumerate(features):
        ax = axes[i]
        style_ax(ax)

        # Build continuous x-axis using timestamps
        before_ts = before['timestamp'].apply(
            lambda x: pd.Timestamp(x).tz_localize(None)).values
        gen_ts = gen['timestamp'].apply(
            lambda x: pd.Timestamp(x).tz_localize(None)).values
        after_ts = after['timestamp'].apply(
            lambda x: pd.Timestamp(x).tz_localize(None)).values

        # Real before
        if col in before.columns:
            ax.plot(before_ts, before[col].values,
                    color=GREEN, linewidth=1.5, alpha=0.9, label='Real')

        # Synthetic fill
        if col in gen.columns:
            ax.plot(gen_ts, gen[col].values,
                    color=BLUE, linewidth=1.2, linestyle='--',
                    alpha=0.9, label='Synthetic Fill')

        # Real after
        if col in after.columns:
            ax.plot(after_ts, after[col].values,
                    color=GREEN, linewidth=1.5, alpha=0.9)

        # Highlight gap region
        gap_start_naive = pd.Timestamp(gap['gap_start']).tz_localize(None)
        gap_end_naive = pd.Timestamp(gap['gap_end']).tz_localize(None)
        ax.axvspan(gap_start_naive, gap_end_naive, color=ORANGE, alpha=0.07)
        ax.axvline(gap_start_naive, color=ORANGE, linewidth=1, linestyle=':', alpha=0.7)
        ax.axvline(gap_end_naive, color=ORANGE, linewidth=1, linestyle=':', alpha=0.7)

        # Transition markers
        if col in before.columns and col in gen.columns and len(gen) > 0:
            last_real = before[col].iloc[-1]
            first_gen = gen[col].iloc[0]
            jump = abs(first_gen - last_real)
            ax.annotate(f'Δ={jump:.2f}',
                       xy=(gap_start_naive, last_real),
                       xytext=(10, 15), textcoords='offset points',
                       color=ORANGE, fontsize=7,
                       arrowprops=dict(arrowstyle='->', color=ORANGE, lw=0.8))

        if col in after.columns and col in gen.columns and len(gen) > 0:
            last_gen = gen[col].iloc[-1]
            first_after = after[col].iloc[0]
            jump = abs(first_after - last_gen)
            ax.annotate(f'Δ={jump:.2f}',
                       xy=(gap_end_naive, first_after),
                       xytext=(10, -15), textcoords='offset points',
                       color=ORANGE, fontsize=7,
                       arrowprops=dict(arrowstyle='->', color=ORANGE, lw=0.8))

        ax.set_ylabel(label, color=TEXT, fontsize=10, fontweight='bold')
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))

        if i == 0:
            real_patch = mpatches.Patch(color=GREEN, alpha=0.9, label='Real Data')
            syn_patch = mpatches.Patch(color=BLUE, alpha=0.9, label='Synthetic Fill')
            gap_patch = mpatches.Patch(color=ORANGE, alpha=0.2, label='Gap Region')
            ax.legend(handles=[real_patch, syn_patch, gap_patch],
                     framealpha=0.2, labelcolor=TEXT, fontsize=9, loc='upper right')

    axes[-1].set_xlabel('Time', color=MUTED, fontsize=10)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(f'gap_fill_{gap["gap_id"]}.png', dpi=150,
                facecolor=BG, bbox_inches='tight')
    plt.show()

    # Print stats
    print(f"\nGap Fill Statistics:")
    for col, label in features:
        if col in before.columns and col in gen.columns and col in after.columns:
            before_mean = before[col].iloc[-5:].mean()
            gen_mean = gen[col].mean()
            after_mean = after[col].iloc[:5].mean()
            gen_std = gen[col].std()

            print(f"  {label:<20} "
                  f"Before(last 5): {before_mean:>8.2f} | "
                  f"Generated mean: {gen_mean:>8.2f} (std: {gen_std:.2f}) | "
                  f"After(first 5): {after_mean:>8.2f}")


In [ ]:
result = fill_gap('pc1', day_data, gaps[3])
plot_filled_gap(result)

In [ ]:
def fill_and_show(device_id, day_data, gap, day_index=0):
    """Fill gap with Mode 2, show before/generated/after rows, save to Excel."""
    
    device_col = f'device_{device_id}'
    device_data = df[df[device_col] == 1].sort_values('timestamp').reset_index(drop=True)
    
    # Context
    before = day_data.iloc[max(0, gap['before_idx'] - 4):gap['before_idx'] + 1]  # last 5 real rows
    after = day_data.iloc[gap['after_idx']:gap['after_idx'] + 5]  # first 5 real rows after gap
    
    # Generate
    start_ts = gap['gap_start'] + pd.Timedelta(seconds=1)
    n_steps = min(gap['duration_sec'] - 1, 3600)
    
    start_lat = before['Latitude'].iloc[-1]
    start_lon = before['Longitude'].iloc[-1]
    
    print(f"Filling Gap {gap['gap_id']}: {n_steps} steps")
    print(f"  Start: ({start_lat:.4f}, {start_lon:.4f}) | Timestamp: {start_ts}")
    
    gen = generate_chain(
        device_id, str(start_ts), n_steps=n_steps,
        mode='with_gps',
        lat=start_lat, lon=start_lon
    )
    
    if gen is None or len(gen) == 0:
        print("Generation failed!")
        return
    
    # Tag source
    before = before.copy()
    gen = gen.copy()
    after = after.copy()
    
    before['source'] = 'REAL_BEFORE'
    gen['source'] = 'GENERATED'
    after['source'] = 'REAL_AFTER'
    
    # Show: 5 before + all generated + 5 after
   
    
    # Combine and save to Excel

    combined = pd.concat([before, gen, after], ignore_index=True)

    for col in combined.columns:
        if hasattr(combined[col], 'dt') and hasattr(combined[col].dt, 'tz') and combined[col].dt.tz is not None:
            combined[col] = combined[col].dt.tz_localize(None)
    
    if hasattr(combined.index, 'tz') and combined.index.tz is not None:
        combined.index = combined.index.tz_localize(None)
    
    

    filename = f'gap_fill_{device_id}_gap{gap["gap_id"]}.csv'
    combined.to_csv(filename, index=False)

   
    

    
    return combined




In [ ]:

day_data, gaps, day_date = find_gaps('pc3', day_index=0)

sorted_gaps = sorted(gaps, key=lambda g: g['duration_sec'])
print(f"Shortest gap: Gap {sorted_gaps[0]['gap_id']} — {sorted_gaps[0]['duration_sec']}s\n")

result = fill_and_show('pc3', day_data, sorted_gaps[0], day_index=0)

In [ ]:
result[['source', 'timestamp','ping_ms', 'datarate', 'jitter', 'Latitude', 'Longitude', 'speed_kmh', 'Altitude', 'PCell_freq_MHz', 'PCell_RSRP_max', 'PCell_RSRQ_max', 'PCell_SNR_1', 'PCell_SNR_2']].head(20)
